# A first de-risked dispatch

We solve the nominal DC dispatch of ACTIVSg200, then de-risk it under $\phi^{\mathrm{joule}}$ at the reference weight $\lambda^{\mathrm{ref}} = z^{0}/\hat{\phi}^{0}$.

Run this notebook from the repository root with an AMPL licence and commit it with its outputs.

## Read the network

In [1]:
from ropf import AlgorithmConfig, read_matpower, run

net = read_matpower("data/case_ACTIVSg200.m")

## The nominal dispatch

At $\lambda = 0$ the master returns the nominal dispatch.

In [2]:
nominal = run(net, AlgorithmConfig(metric="joule_loss_max", weight_multiplier=0))
nominal.z0, nominal.rho0, nominal.lambda_ref

(27479.64330609605, 0.012455402506202157, 2206242.8968002107)

## The de-risked dispatch

`weight_multiplier = 1` prices the surrogate at $\lambda^{\mathrm{ref}}$. The log tags each step with the line of Algorithm 1 it implements; `alg1_lines` drops the solver's own output.

In [3]:
def alg1_lines(message):
    for line in message.splitlines():
        if line.strip() and not line.startswith("  ") and not line.startswith("master.mod"):
            print(line)

result = run(net, AlgorithmConfig(metric="joule_loss_max", weight_multiplier=1, gamma_tol=1e-6),
             log=alg1_lines)

baseline: joule_loss_max (q = 1), lambda = 1 lambda_ref, kappa = 5, eta = 0.85, k_bar = 40


ALG1-L1  nominal solve


 solved, cost 27479.643306, Phi 0.000000, 0.5s
ALG1-L2  z0 = 27479.643306, rho0 = 0.012455
lambda_ref = 2206242.896800, lambda = 2206242.896800
ALG1-L7  k = 1, 5 line + 0 bus cuts


 solved, cost 30019.185972, Phi 0.001218, 0.5s
ALG1-L8  rho = 0.002855, Phi = 0.001218, Gamma = 5.735e-01, rho/rho0 = 0.229257
ALG1-L7  k = 2, 10 line + 0 bus cuts
 solved, cost 29280.994011, Phi 0.002324, 0.5s
ALG1-L8  rho = 0.002324, Phi = 0.002324, Gamma = -1.636e-11, rho/rho0 = 0.186619
converged: Gamma = -1.636e-11 at k = 2 (1.8s)


## Iterations

In [4]:
for it in result.iterations:
    print(f"{it.k:3d} {it.cost:14.2f} {it.surrogate:10.5f} {it.rho:10.5f} {it.gamma:9.2e}")
result.termination

  0       27479.64    0.00000    0.01246  1.00e+00
  1       30019.19    0.00122    0.00286  5.73e-01
  2       29280.99    0.00232    0.00232 -1.64e-11


'converged'

## Overhead and exposure reduction

In [5]:
overhead = result.final.cost / result.z0 - 1
reduction = 1 - result.final.rho / result.rho0
overhead, reduction

(0.06555218656037187, 0.813380990390109)